# 2a. Provider catalog, policies and connectors

The Provider publishes what it offers as a DCAT catalog: datasets, their distributions (how the
data is delivered) and ODRL policies (the terms under which it may be used). A connector instance
bound to each distribution tells the Provider how to reach the real backend when a transfer
starts; in this notebook the connectors use no authentication (`NO_AUTH`).

Steps:
1. Onboarding (`setup_auth.ipynb`), guarded.
2. Resolve the Provider's main catalog and data service (`/api/v1/catalog-agent/catalogs/main`, `/data-services/main`).
3. Create a dataset with a PULL distribution (`http+pull`) and a PUSH distribution (`http+push`) (`/api/v1/catalog-agent/datasets`, `/distributions`).
   Then ask the doctor whether it is DSP, DCAT 3 and DCAT-AP compliant (`/datasets/{id}/doctor`) and read the public DCAT-AP catalog (`/public/dcat-ap/catalog`).
4. Attach ODRL offers instantiated and composed from policy templates (`/api/v1/catalog-agent/policy-templates`, `/instantiate-odrl-offer`, `/odrl-policies/compose`).
5. Attach two ODRL offers written by hand and list the dataset's offers (`/api/v1/catalog-agent/odrl-policies`, `/odrl-policies/entity/{id}`).
6. Register a PULL connector template, instantiate it on the PULL distribution and inspect it (`/api/v1/connector/templates`, `/instances`, `/instances/distribution/{id}`).
7. Register a PUSH connector template, instantiate it on the PUSH distribution and inspect it.
8. The Consumer discovers the catalog and dataset over DSP (`/dsp/current/catalog/rpc/setup-catalog-request`, `setup-dataset-request`) and inspects the cached peer catalog (`/api/v1/catalog-agent/peer-catalogs/{peerId}`).

**Runs on its own.** It leaves `target_id`, `policy_id` and `data_space_provider_dsp_endpoint`
ready for the negotiation notebooks, and sets `catalog_setup = "no_auth"` so that the transfer
notebooks know which variant is loaded. Each run creates a new dataset. `setup_catalog_policies_oauth.ipynb` is the
same flow with connectors that authenticate with a bearer token.

## 1. Prerequisites

The guard runs the onboarding notebook only when this kernel has not run it yet. `%run` on a
notebook executes its cells in this same namespace, so the URLs, DIDs and `show` it defines are
available below.

In [1]:
if not globals().get("onboarded"):
    %run ./setup_auth.ipynb

Authority URL: http://127.0.0.1:1500
Consumer URL:  http://127.0.0.1:1100
Provider URL:  http://127.0.0.1:1200
Wallet linked for Authority (http://127.0.0.1:1500): wallet id=fafnir-local, name=fafnir-wallet
Wallet linked for Consumer (http://127.0.0.1:1100): wallet id=fafnir-local, name=fafnir-wallet
Wallet linked for Provider (http://127.0.0.1:1200): wallet id=fafnir-local, name=fafnir-wallet
HTTP 200 GET http://127.0.0.1:1100/api/v1/mates/myself
{
  "participant_id": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ5cS12NXdJNFM2MTl2SnJITmpENDVra0owUGw4VGlBTzZYTnFJc1o4SDBaTEs5T2JoWTBTeGtEQ2JlZWd2RlZEU0MzM182N3lVZVpUWFI2SjFXUExHdHNtLU51SXJtR1RVRVgteFA0WnFLVGRFUTFSdUV0MExtMUhtdkdJWG1zLVJrTXRNVjcxRWt0MjNiZFYzX01WTjRUOUd5WnlaQTB6RDRlTFUxRTRTRzlRMGl4MGFRbC16eGdsVFRYQ2l4VTdCTzRXcHRkaUp3a0Jid2hMbnVqeXFzSmNvQXpPU000aXZtZlRTVW1Yd0Nrc1gySTFxVTNnVWdtUGFtbmEta3ZUb3FYZUk4ODNlRTFkSWNuUG53dmJlRmpfa0xuQ2JrNzhjNkVkRXZ0OVBMa3ByTWZfdmVxeVlOWjB0WlpYUGM5c0lRVlMyUG5xZkN5RUhsWWVuMXNXclEifQ",
  "participant_n

## 2. Main catalog and data service (`/api/v1/catalog-agent/*`)

Every resource of the catalog is JSON-LD in the agent's own context
(`https://w3id.org/eunomia/v1/catalog-context.jsonld`): `@id` is its URN, texts are language maps
and version, visibility and owner travel in headers (`ETag`, `Eunomia-Visibility`). Every Provider
starts with a `main` catalog and a `main` data service. The data service is the
Provider's DSP endpoint; distributions point to it through `accessService`.

In [ ]:
catalog = show(requests.get(f"{data_space_provider}/api/v1/catalog-agent/catalogs/main", headers=json_header))
catalog_id = catalog["@id"]

In [ ]:
data_service = show(requests.get(f"{data_space_provider}/api/v1/catalog-agent/data-services/main", headers=json_header))
data_service_id = data_service["@id"]

## 3. Dataset and distributions (`/api/v1/catalog-agent/datasets`, `/distributions`)

A dataset is the unit of negotiation: a contract agreement always targets a dataset. Its
distributions are the ways to obtain it. The format of a distribution is the `format` a transfer
request asks for later:
- `http+pull`: the Consumer pulls the data from an endpoint the Provider hands over.
- `http+push`: the Provider pushes the data to a sink the Consumer names.

In [ ]:
# A Dataset in the catalog's JSON-LD context: texts are language maps, `inCatalog` places it.
payload = {
    "@context": "https://w3id.org/eunomia/v1/catalog-context.jsonld",
    "@type": "Dataset",
    "title": {"en": "My dataspace dataset", "es": "Mi dataset del espacio de datos"},
    "description": {"en": "Readings shared through the dataspace"},
    "keyword": {"en": ["tutorial", "readings"]},
    "theme": ["http://publications.europa.eu/resource/authority/data-theme/TECH"],
    "inCatalog": catalog_id,
}
dataset = show(requests.post(f"{data_space_provider}/api/v1/catalog-agent/datasets", headers=json_header, json=payload))
dataset_id = dataset["@id"]

In [ ]:
# `format` is the DSP transfer format (`dct:format`); `ofDataset` places it under the dataset.
payload = {
    "@context": "https://w3id.org/eunomia/v1/catalog-context.jsonld",
    "@type": "Distribution",
    "title": {"en": "Distribution of my dataset"},
    "description": {"en": "Pull access to my dataset"},
    "accessService": [data_service_id],
    "ofDataset": dataset_id,
    "format": "http+pull",
}
distribution = show(requests.post(f"{data_space_provider}/api/v1/catalog-agent/distributions", headers=json_header, json=payload))
distribution_pull_id = distribution["@id"]

In [ ]:
payload = {
    "@context": "https://w3id.org/eunomia/v1/catalog-context.jsonld",
    "@type": "Distribution",
    "title": {"en": "Distribution of a streaming dataset"},
    "description": {"en": "Push delivery of my streaming dataset"},
    "accessService": [data_service_id],
    "ofDataset": dataset_id,
    "format": "http+push",
}
distribution = show(requests.post(f"{data_space_provider}/api/v1/catalog-agent/distributions", headers=json_header, json=payload))
distribution_push_id = distribution["@id"]

dataset_distributions = show(requests.get(f"{data_space_provider}/api/v1/catalog-agent/distributions/dataset/{dataset_id}", headers=json_header))
print(f"Distributions on dataset {dataset_id}: {[d['format'] for d in dataset_distributions['@graph']]}")

## 3b. Is the dataset DSP, DCAT 3 and DCAT-AP compliant? (`/datasets/{id}/doctor`, `/public/{profile}`)

The catalog stores one neutral RDF description and projects it per profile. The doctor asks every
registered profile whether the resource conforms and why not; `?deep=true` checks the dataset with
its distributions, `?profile=dcat-ap` asks one profile only. Resources marked `Public` are also
served without authentication in each DCAT profile under `/api/v1/catalog-agent/public/{profile}/`.

In [ ]:
report = show(requests.get(f"{data_space_provider}/api/v1/catalog-agent/datasets/{dataset_id}/doctor?deep=true", headers=json_header))
for profile in report["profiles"]:
    print(f"{profile['profile']:8} compliant={profile['compliant']} violations={profile['violations']} warnings={profile['warnings']}")
    for finding in profile["findings"][:5]:
        print(f"   {finding['severity']:9} {finding.get('property', '')}: {finding['message']}")

In [ ]:
# The main catalog is the connector's and public: anyone reads it in DCAT-AP, no token needed.
dcat_ap = show(requests.get(f"{data_space_provider}/api/v1/catalog-agent/public/dcat-ap/catalog"))
print(f"DCAT-AP graph: {len(dcat_ap['@graph'])} nodes, profile {requests.get(f'{data_space_provider}/api/v1/catalog-agent/public/dcat-ap/catalog').headers.get('Link')}")

## 4. ODRL offers from policy templates (`/api/v1/catalog-agent/policy-templates`)

Policy templates are reusable ODRL pieces with `$parameters`. The agent seeds a library
(`static/policy_templates`): constraints such as `expires-after` or `participant-allowlist`,
rules such as `no-redistribution`, and policies built from them such as `research-30-days`,
which references `purpose`, `expires-after` and `no-redistribution` with `@ref`. Each template
says where it is meant to apply (`controlPoints`); nothing evaluates policies yet.

Instantiating a policy template fills its parameters, expands its references and attaches the
plain ODRL to the dataset. Composing (`/odrl-policies/compose`) adds several parts up: here the
research policy for a week, with a constraint that joins its permission and lets only UPM in.


In [ ]:
templates = show(requests.get(f"{data_space_provider}/api/v1/catalog-agent/policy-templates?kind=policy", headers=json_header))

In [ ]:
payload = {
    "id": "research-30-days",
    "version": "1.0",
    "parameters": {
        "$duration": "P30D"
    },
    "entityId": dataset_id,
    "entityType": "Dataset",
    "description": "Research use for 30 days after the agreement, without redistribution.",
}
policy = show(requests.post(f"{data_space_provider}/api/v1/catalog-agent/policy-templates/instantiate-odrl-offer", headers=json_header, json=payload))

In [ ]:
payload = {
    "entityId": dataset_id,
    "entityType": "Dataset",
    "description": "Research use by UPM for a week.",
    "parts": [
        {"@ref": "research-30-days@1.0", "with": {"$duration": "P7D"}},
        {"@ref": "participant-allowlist@1.0", "with": {"$participants": ["did:web:university.upm.es"]}, "into": "permission"},
    ],
}
composed = show(requests.post(f"{data_space_provider}/api/v1/catalog-agent/odrl-policies/compose", headers=json_header, json=payload))

## 5. ODRL offers written by hand (`/api/v1/catalog-agent/odrl-policies`)

An offer can also be posted as plain ODRL: permissions, obligations (duties) and prohibitions,
each with optional constraints. `GET /api/v1/catalog-agent/odrl-policies/entity/{dataset_id}` lists
all offers attached to the dataset.

In [ ]:
payload = {
    "odrlOffer": {
        "@context": "http://www.w3.org/ns/odrl.jsonld",
        "@type": "Offer",
        "permission": [{
            "action": "use",
            "constraint": [
                {"leftOperand": "dateTime", "operator": "gteq", "rightOperand": "2026-01-01T00:00:00Z"},
                {"leftOperand": "dateTime", "operator": "lteq", "rightOperand": "2026-12-31T23:59:59Z"},
                {"leftOperand": "purpose", "operator": "isA", "rightOperand": "commercial"},
            ],
        }],
        "obligation": [{
            "action": "attribute",
            "constraint": [{"leftOperand": "recipient", "operator": "isA", "rightOperand": "public"}],
        }],
        "prohibition": [{"action": "derive", "constraint": []}],
    },
    "entityId": dataset_id,
    "entityType": "Dataset",
    "description": "Commercial licence for 2026: commercial use with mandatory attribution; derivation is prohibited.",
}
policy = show(requests.post(f"{data_space_provider}/api/v1/catalog-agent/odrl-policies", headers=json_header, json=payload))

In [ ]:
payload = {
    "odrlOffer": {
        "@context": "http://www.w3.org/ns/odrl.jsonld",
        "@type": "Offer",
        "permission": [{
            "action": "read",
            "constraint": [
                {"leftOperand": "dateTime", "operator": "gteq", "rightOperand": "2026-03-01T00:00:00Z"},
                {"leftOperand": "dateTime", "operator": "lteq", "rightOperand": "2026-09-01T00:00:00Z"},
                {"leftOperand": "purpose", "operator": "isA", "rightOperand": "research"},
                {"leftOperand": "count", "operator": "lteq", "rightOperand": "10000"},
            ],
        }],
        "obligation": [],
        "prohibition": [
            {"action": "distribute", "constraint": []},
            {"action": "reproduce", "constraint": []},
        ],
    },
    "entityId": dataset_id,
    "entityType": "Dataset",
    "description": "Research trial (Mar-Sep 2026): read only, up to 10 000 requests, research purposes; no redistribution or reproduction.",
}
policy = show(requests.post(f"{data_space_provider}/api/v1/catalog-agent/odrl-policies", headers=json_header, json=payload))

dataset_policies = show(requests.get(f"{data_space_provider}/api/v1/catalog-agent/odrl-policies/entity/{dataset_id}", headers=json_header))
print(f"Total policies on dataset {dataset_id}: {len(dataset_policies)}")

## 6. PULL connector (`/api/v1/connector/*`)

A connector template describes how to reach a backend, with `{{__NAME__}}` placeholders; an
instance fills the parameters and binds the template to a distribution. In PULL mode the template
describes the data access the Consumer gets once the transfer starts.
`GET /api/v1/connector/instances/distribution/{id}` retrieves the instance bound to a distribution.

In [ ]:
payload = {
    "@context": "https://w3id.org/eunomia/v1/connector-context.jsonld",
    "@type": "ConnectorTemplate",
    "authentication": {"type": "NO_AUTH"},
    "interaction": {
        "mode": "PULL",
        "dataAccess": {
            "protocol": "HTTP",
            "urlTemplate": "{{__ACCESS_URL__}}",
            "method": "{{__ACCESS_METHODS__}}",
            "headers": "{{__ACCESS_HEADERS__}}"
        }
    },
    "parameters": [
        {"paramType": "STRING", "name": "ACCESS_URL", "title": "Url", "required": True},
        {"paramType": "VEC<STRING>", "name": "ACCESS_METHODS", "title": "Methods", "required": True},
        {"paramType": "MAP<STRING,STRING>", "name": "ACCESS_HEADERS", "title": "Headers", "required": True}
    ]
}
template = show(requests.post(f"{data_space_provider}/api/v1/connector/templates", headers=json_header, json=payload))

In [ ]:
payload = {
    "@context": "https://w3id.org/eunomia/v1/connector-context.jsonld",
    "@type": "ConnectorInstance",
    "templateName": template["name"],
    "templateVersion": template["version"],
    "distributionId": distribution_pull_id,
    "metadata": {
        "description": "Pull access to the demo backend",
        "ownerId": "tutorial"
    },
    "dryRun": False,
    "parameters": {
        "ACCESS_URL": "http://127.0.0.1:8081",
        "ACCESS_METHODS": ["GET", "HEAD"],
        "ACCESS_HEADERS": {"Content-type": "application/json"}
    }
}
instance = show(requests.post(f"{data_space_provider}/api/v1/connector/instances", headers=json_header, json=payload))
pull_connector = show(requests.get(f"{data_space_provider}/api/v1/connector/instances/distribution/{distribution_pull_id}", headers=json_header))

## 7. PUSH connector (`/api/v1/connector/*`)

In PUSH mode the template says how to subscribe the Provider to the backend when a transfer
starts, and how to unsubscribe when it stops. `SUB_URL` and `UNSUB_URL` are filled by the
instance; the `SYS_*` and `RUNTIME_*` placeholders are filled by the agent at run time (its own
URL, the ingress of the transfer, and the subscription id the backend answered).

In [ ]:
payload = {
    "@context": "https://w3id.org/eunomia/v1/connector-context.jsonld",
    "@type": "ConnectorTemplate",
    "authentication": {"type": "NO_AUTH"},
    "interaction": {
        "mode": "PUSH",
        "subscribe": {
            "protocol": "HTTP",
            "urlTemplate": "{{__SUB_URL__}}",
            "method": "POST",
            "headers": {},
            "bodyTemplate": json.dumps({
                "url": "{{__SYS_OWN_URL_DOCKER__}}{{__RUNTIME_INGRESS__}}",
                "event_type": "hotel_waste_generated"
            })
        },
        "unsubscribe": {
            "protocol": "HTTP",
            "urlTemplate": "{{__UNSUB_URL__}}/{{__RUNTIME_JSON_{.subscription.data.ID}__}}",
            "method": "POST",
            "headers": {},
            "bodyTemplate": json.dumps({})
        }
    },
    "parameters": [
        {"paramType": "STRING", "name": "SUB_URL", "title": "Subscription URL", "required": True},
        {"paramType": "STRING", "name": "UNSUB_URL", "title": "Unsubscription URL", "required": True}
    ]
}
template = show(requests.post(f"{data_space_provider}/api/v1/connector/templates", headers=json_header, json=payload))

In [ ]:
payload = {
    "@context": "https://w3id.org/eunomia/v1/connector-context.jsonld",
    "@type": "ConnectorInstance",
    "templateName": template["name"],
    "templateVersion": template["version"],
    "distributionId": distribution_push_id,
    "parameters": {
        "SUB_URL": "http://127.0.0.1:8082/subscriptions/subscribe",
        "UNSUB_URL": "http://127.0.0.1:8082/subscriptions/unsubscribe"
    }
}
instance = show(requests.post(f"{data_space_provider}/api/v1/connector/instances", headers=json_header, json=payload))
push_connector = show(requests.get(f"{data_space_provider}/api/v1/connector/instances/distribution/{distribution_push_id}", headers=json_header))

## 8. Catalog and dataset discovery by the Consumer (DSP 6.2.1, DSP 6.2.2 & `/api/v1/catalog-agent/peer-catalogs`)

The Consumer asks its own agent to fetch the Provider's catalog over DSP
(`POST /dsp/current/catalog/rpc/setup-catalog-request`, sending `CatalogRequestMessage`) and can
also request a single dataset (`POST /dsp/current/catalog/rpc/setup-dataset-request`, sending
`DatasetRequestMessage`). When fetched with `noCache: false`, the catalog is also cached and
exposed under `GET /api/v1/catalog-agent/peer-catalogs/{peerId}`.

From the answer we keep:
- `target_id`: the `@id` of the dataset created above.
- `policy_id`: the `@id` of one of its offers. A negotiation must open on an offer the catalog
  publishes, on the dataset that publishes it.
- `data_space_provider_dsp_endpoint`: the `endpointURL` of the Provider's data service (`service`
  is a list).

In [ ]:
payload = {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcCatalogRequestMessage",
    "associatedAgentPeer": data_space_provider_participant_id,
    "noCache": True,
}
provider_catalog = show(requests.post(f"{data_space_consumer}/dsp/current/catalog/rpc/setup-catalog-request", headers=json_header, json=payload))["response"]

In [ ]:
dataset_rpc = show(requests.post(
    f"{data_space_consumer}/dsp/current/catalog/rpc/setup-dataset-request",
    headers=json_header,
    json={
        "@context": "https://w3id.org/eunomia/v1/context.jsonld",
        "@type": "RpcDatasetRequestMessage",
        "associatedAgentPeer": data_space_provider_participant_id,
        "dataset": dataset_id,
    },
))["response"]

published = next(d for d in provider_catalog["dataset"] if d["@id"] == dataset_id)
assert dataset_rpc["@id"] == published["@id"]

target_id = published["@id"]
policy_id = published["hasPolicy"][0]["@id"]
data_space_provider_dsp_endpoint = provider_catalog["service"][0]["endpointURL"]

# Marker read by the guards of the negotiation and transfer notebooks
catalog_setup = "no_auth"

print(f"Dataset (target):      {target_id}")
print(f"Offer (policy):        {policy_id}")
print(f"Provider DSP address:  {data_space_provider_dsp_endpoint}")